# 종합 실습 — 주문·고객·상품 통합 분석

## 1. 데이터 준비
`np.random.seed(7)` 스냅샷 셀을 그대로 사용하고, 과제 명칭에 맞춰 `orders`·`customers`·`products`로 이름을 붙입니다.

In [18]:
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
# 종합 실습용 데이터 — 새 스냅샷 (이 셀부터 단독 실행 가능)
# ─────────────────────────────────────────────
np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})
print("스냅샷 준비 완료 — orders:", ordr.shape, "| customers:", cust.shape, "| products:", prod.shape)

# 과제 명칭으로 정리 (원본 보존을 위해 copy)
orders, customers, products = ordr.copy(), cust.copy(), prod.copy()

스냅샷 준비 완료 — orders: (1500, 6) | customers: (200, 3) | products: (30, 3)


## 2. 키 검증 (병합 전 필수)
세 가지를 확인합니다.
1. **중복** — 차원 테이블(`customers`, `products`)의 키가 유일한가? (아니면 병합 시 행이 불어남)
2. **미매칭** — `orders`의 외래키가 모두 차원 테이블에 존재하는가? (아니면 NaN 발생)
3. **관계** — 주문 : 고객, 주문 : 상품이 실제로 m:1 인가?

In [19]:
# ① 중복 검사
dup = pd.Series({
    "orders.order_id": orders["order_id"].duplicated().sum(),
    "customers.customer_id": customers["customer_id"].duplicated().sum(),
    "products.product_id": products["product_id"].duplicated().sum(),
}, name="중복 수")
print("① 중복 검사\n", dup, "\n")

# ② 미매칭 검사 (orders 쪽 키가 차원 테이블에 없는 경우)
unmatched = pd.Series({
    "orders.customer_id → customers": (~orders["customer_id"].isin(customers["customer_id"])).sum(),
    "orders.product_id → products":   (~orders["product_id"].isin(products["product_id"])).sum(),
}, name="미매칭 수")
print("② 미매칭 검사\n", unmatched, "\n")

# ③ 관계 검사: 차원 키가 유일(1) & 주문 쪽은 여러 번 등장(m)
print("③ 관계 검사")
print("  customers 키 유일:", customers["customer_id"].is_unique,
      "| 고객당 주문 수 최대:", orders.groupby("customer_id").size().max())
print("  products  키 유일:", products["product_id"].is_unique,
      "| 상품당 주문 수 최대:", orders.groupby("product_id").size().max())

assert dup.sum() == 0 and unmatched.sum() == 0, "키 검증 실패 — 병합 전에 정제 필요"

① 중복 검사
 orders.order_id          0
customers.customer_id    0
products.product_id      0
Name: 중복 수, dtype: int64 

② 미매칭 검사
 orders.customer_id → customers    0
orders.product_id → products      0
Name: 미매칭 수, dtype: int64 

③ 관계 검사
  customers 키 유일: True | 고객당 주문 수 최대: 16
  products  키 유일: True | 상품당 주문 수 최대: 64


### 검증 결과 기록
| 검증 | 결과 |
|---|---|
| ① 중복 | `order_id`·`customer_id`·`product_id` 모두 중복 **0건** |
| ② 미매칭 | 주문의 고객키·상품키 모두 미매칭 **0건** (1,500건 전부 매칭) |
| ③ 관계 | 고객·상품 키는 유일(1), 고객당 주문 최대 16건·상품당 주문 다수(m) → **m:1** |

**검증 결과 차원 테이블 키에 중복이 없고 주문 키가 모두 매칭되며 관계가 주문(m) : 고객/상품(1) 이므로 m:1 병합이 안전하다.**

## 3. 병합 (`validate` 적용)
`validate="m:1"`로 관계를 강제하고, `indicator`로 매칭 여부를 한 번 더 확인합니다.

In [20]:
df = orders.merge(customers, on="customer_id", how="left",
                  validate="m:1", indicator="_m_cust")
df = df.merge(products, on="product_id", how="left",
              validate="m:1", indicator="_m_prod")

print(df["_m_cust"].value_counts(), "\n")
print(df["_m_prod"].value_counts(), "\n")

# 병합 후 사후 점검: 행 수·매출 합계 보존
assert len(df) == len(orders), "행 수가 변했습니다!"
assert df["amount"].sum() == orders["amount"].sum(), "매출 합계가 변했습니다!"
df = df.drop(columns=["_m_cust", "_m_prod"])
print("병합 완료:", df.shape)
df.head()

_m_cust
both          1500
left_only        0
right_only       0
Name: count, dtype: int64 

_m_prod
both          1500
left_only        0
right_only       0
Name: count, dtype: int64 

병합 완료: (1500, 10)


,order_id,customer_id,product_id,quantity,amount,order_date,region,membership,category,price
0,O00001,C0032,P005,3,120000.0,2025-04-12,서울,premium,패션,40000
1,O00002,C0049,P020,1,25000.0,2025-03-09,서울,basic,식품,25000
2,O00003,C0041,P016,3,36000.0,2025-01-27,인천,basic,패션,12000
3,O00004,C0123,P002,3,120000.0,2025-03-26,경기,basic,식품,40000
4,O00005,C0150,P014,3,120000.0,2025-03-16,경기,basic,가전,40000


## 4. 집계
### 4-1. 지역 × 회원등급 매출 교차표 (`pivot_table`)

In [21]:
region_mem = df.pivot_table(
    index="region", columns="membership", values="amount",
    aggfunc="sum", fill_value=0, margins=True, margins_name="합계",
)
region_mem = region_mem[["basic", "premium", "vip", "합계"]]
print("지역 × 회원등급 매출 (단위: 만원)")
(region_mem / 10_000).round(0).astype(int)

지역 × 회원등급 매출 (단위: 만원)


membership,basic,premium,vip,합계
region,,,,
경기,1459,323,332,2114
부산,1153,486,237,1875
서울,1191,924,232,2348
인천,1737,968,53,2758
합계,5541,2700,854,9096


In [22]:
# 해석 보조: 고객 수와 1인당 매출
n_cust_tbl = df.pivot_table(index="region", columns="membership", values="customer_id",
                            aggfunc="nunique", margins=True, margins_name="합계")
per_cust = (region_mem / n_cust_tbl / 10_000).round(1)
print("고객 수\n", n_cust_tbl, "\n")
print("1인당 매출(만원)\n", per_cust)

고객 수
 membership  basic  premium  vip   합계
region                              
경기             31        8    6   45
부산             24       12    5   41
서울             31       16    6   53
인천             38       22    1   61
합계            124       58   18  200 

1인당 매출(만원)
 membership  basic  premium   vip    합계
region                                
경기           47.1     40.3  55.4  47.0
부산           48.0     40.5  47.3  45.7
서울           38.4     57.8  38.7  44.3
인천           45.7     44.0  53.0  45.2
합계           44.7     46.6  47.5  45.5


### 4-2. 카테고리별 KPI 요약표 (`groupby` + `agg`)

In [23]:
cat_kpi = (
    df.groupby("category")
      .agg(매출=("amount", "sum"),
           건수=("order_id", "count"),
           객단가=("amount", "mean"))
      .sort_values("매출", ascending=False)
)
cat_kpi["매출비중"] = cat_kpi["매출"] / cat_kpi["매출"].sum()
cat_kpi.round({"객단가": 0, "매출비중": 3})

,매출,건수,객단가,매출비중
category,,,,
패션,38318000.0,512,74840.0,0.421
식품,22404000.0,398,56291.0,0.246
가전,20248000.0,398,50874.0,0.223
뷰티,9986000.0,192,52010.0,0.110


## 5. 특성 추가 (`transform`)
`transform("sum")`은 그룹 합계를 **원래 행 길이 그대로** 돌려주므로, 행 단위 나눗셈이 바로 됩니다.

In [24]:
df["region_share"] = df["amount"] / df.groupby("region")["amount"].transform("sum")

# 검증: 지역별 비중 합은 정확히 1이어야 함
print(df.groupby("region")["region_share"].sum().round(10))
df.sort_values("region_share", ascending=False)[
    ["order_id", "region", "category", "amount", "region_share"]].head()

region
경기    1.0
부산    1.0
서울    1.0
인천    1.0
Name: region_share, dtype: float64


,order_id,region,category,amount,region_share
1278,O01279,부산,식품,225000.0,0.011997
1286,O01287,부산,식품,225000.0,0.011997
1395,O01396,부산,패션,225000.0,0.011997
1087,O01088,부산,패션,225000.0,0.011997
324,O00325,부산,식품,225000.0,0.011997


## 6. 보고서

**분석 목적** 지역·회원등급·카테고리별 매출 구조를 파악해 우선 개선 영역을 찾는다.

**데이터** 주문 1,500건 · 고객 200명 · 상품 30개 (2025-01-01 ~ 2025-05-30), 총매출 **9,095.6만원**

**방법** 키 검증(중복·미매칭·관계) → `validate="m:1"` 병합 → `pivot_table`·`groupby.agg` 집계 → `transform` 비중 컬럼 추가

### 발견 1 — 패션이 매출의 42%를 차지하고 객단가도 가장 높다
패션 매출 3,831.8만원(42.1%), 객단가 74,840원으로 2위 식품(56,291원)보다 약 33% 높다. 상품 수가 가장 많고(10개) 40,000·75,000원대 상품 비중이 높은 구성 덕분이다. 반면 뷰티는 상품이 4개뿐이며 매출 비중 11.0%, 건수 192건으로 가장 작다.

### 발견 2 — 지역 매출 차이는 '고객 수' 차이이지 '고객 가치' 차이가 아니다
인천이 매출 1위(2,757.7만원, 30.3%)지만 고객도 가장 많다(61명). 1인당 매출은 경기 47.0 · 부산 45.7 · 인천 45.2 · 서울 44.3만원으로 지역 간 차이가 6% 이내다. 객단가는 경기가 66,078원으로 가장 높다.

### 발견 3 — VIP의 1인당 매출이 일반 등급과 거의 다르지 않다
1인당 매출은 basic 44.7 · premium 46.6 · vip 47.5만원으로, VIP가 basic보다 약 6% 높을 뿐이다. 또한 인천은 VIP가 단 1명(매출 53만원)이라 지역 × 등급 교차표에서 해당 칸이 사실상 개인 값이다. 서울 premium(16명)의 1인당 매출이 57.8만원으로 교차표 내 최고다.

### 제언 — 회원등급 혜택을 '실제 구매 성과'에 연동하도록 재설계 검토
현재 등급이 구매력 차이를 거의 설명하지 못하므로, 등급 부여 기준을 누적 구매액·구매 빈도 기반으로 바꾸고 VIP 혜택 비용 대비 효과를 점검할 것을 제안한다. 우선 성과가 높은 서울 premium 고객군을 대상으로 상위 등급 전환 프로모션을 시범 운영해 반응을 측정한다.

**한계** 이 데이터는 `np.random`으로 균등하게 생성된 합성 데이터이므로 위 차이 상당 부분은 무작위 변동일 수 있다. 특히 인천 VIP(1명)처럼 표본이 작은 칸은 해석에서 제외해야 하며, 실데이터라면 기간 확대·통계 검정이 필요하다.